# Positive-P Method

This notebook runs the cleaned single-site positive-P method for pure dissipation.

Current scope:
- one site only
- no hopping `J`
- supports `fock` and `coherent` initial states
- uses Olsen-style `alpha` and `alphaPlus` internally
- relation to Deuar notation: `alphaTilde = conj(alphaPlus)`

Important note about optimization:
- if `interaction_strength = 0`, the code switches to a deterministic fast path and `num_of_samples` is ignored
- if `interaction_strength != 0`, the code uses a vectorized Positive-P trajectory template and batches all samples at once instead of parallelizing a Python `for` loop
- batching all samples together makes CPU execution faster and also allows optional GPU acceleration through `cupy`

Example filename:
- `positiveP_fock_numOfParticles1_gamma1p0_time5p0_dt0p001_numOfSamples30000.csv`


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
import matplotlib.pyplot as plt

try:
    import scienceplots  # noqa: F401
    plt.style.use(["science"])
    USING_SCIENCEPLOTS = True
except ImportError:
    USING_SCIENCEPLOTS = False
    print("scienceplots is not installed; using default matplotlib style.")

from bosonic_dissipation.positive_p_method import run_positive_p_and_save

positive_p_config = {
    "initial_state_type": "fock",
    "num_of_particles": 1,
    "interaction_strength": 0.0,
    "gamma": 1.0,
    "time": 5.0,
    "dt": 1e-3,
    "num_of_samples": 30000,
    "prefer_gpu": True,
    "seed": 123,
}

print(f"Using scienceplots: {USING_SCIENCEPLOTS}")
positive_p_config

## Runtime Block: Positive-P

This block produces the real mean particle number and real variance for the project CSV, while also keeping the imaginary parts available for diagnostics.

Internal estimator convention:
- mean particle number is computed from `mean(alpha * alphaPlus)`
- if you rewrite in Deuar notation with `alphaTilde = conj(alphaPlus)`, that becomes `mean(alpha * conj(alphaTilde))`


In [ ]:
positive_p_result, positive_p_csv_path = run_positive_p_and_save(
    OUTPUT_DIR,
    initial_state_type=positive_p_config["initial_state_type"],
    num_of_particles=positive_p_config["num_of_particles"],
    interaction_strength=positive_p_config["interaction_strength"],
    gamma=positive_p_config["gamma"],
    time=positive_p_config["time"],
    dt=positive_p_config["dt"],
    num_of_samples=positive_p_config["num_of_samples"],
    prefer_gpu=positive_p_config["prefer_gpu"],
    seed=positive_p_config["seed"],
)

print(f"Backend used: {positive_p_result.backend}")
print(f"Runtime (s): {positive_p_result.runtime_seconds:.6f}")
print(f"Fast path used: {positive_p_result.fast_path_used}")
print(positive_p_result.notes)
print(f"Saved CSV: {positive_p_csv_path}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

label_suffix = positive_p_result.initial_state_type.capitalize()

axes[0].plot(
    positive_p_result.time_values,
    positive_p_result.mean_particle_number,
    label=f"Positive-P ({label_suffix})",
)
axes[0].set_title("Mean Particle Number")
axes[0].set_xlabel("Time")
axes[0].set_ylabel("Mean Particle Number")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(
    positive_p_result.time_values,
    positive_p_result.variance,
    label=f"Positive-P ({label_suffix})",
    color="tab:orange",
)
axes[1].set_title("Variance")
axes[1].set_xlabel("Time")
axes[1].set_ylabel("Variance")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

In [ ]:
positive_p_result.time_values[:5], positive_p_result.mean_particle_number[:5], positive_p_result.variance[:5]

In [ ]:
positive_p_result.mean_particle_number_imag[:5], positive_p_result.variance_imag[:5]